In [ ]:
import os
from tqdm import tqdm
from dotenv import load_dotenv
from utils import get_stock_code_list, fetch_baostock_data
import clickhouse_connect
import pandas as pd

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS quant

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS quant.stock_daily_market
(
    `trade_date`   Date      COMMENT '交易日',
    `stock_code`   String    COMMENT '不带市场标识的股票代码',
    `open`         Float64   COMMENT '开盘价',
    `close`        Float64   COMMENT '收盘价',
    `high`         Float64   COMMENT '最高价',
    `low`          Float64   COMMENT '最低价',
    `volume`       Int64     COMMENT '成交量（单位：手）',
    `amount`       Float64   COMMENT '成交额（单位：元）',
    `pct_chg`      Float64   COMMENT '涨跌幅（单位：%）',
    `turnover_rate` Float64  COMMENT '换手率（单位：%）'
)
ENGINE = ReplacingMergeTree
PARTITION BY toYYYYMM(trade_date)
ORDER BY (stock_code, trade_date)
SETTINGS index_granularity = 8192;

In [ ]:
load_dotenv()
host = os.getenv('CH_HOST')
password = os.getenv('CH_PASSWORD')

client = clickhouse_connect.get_client(
    host=host,
    port=8123,
    username='default',
    password=password,
    database='quant'
)

In [ ]:
stock_code_lst = get_stock_code_list(filter_exchange=True)
start_date, end_date = "2010-01-01", "2018-01-01"

column_mapping = {
    'date': 'trade_date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Amount': 'amount',
    'pctChg': 'pct_chg',
    'turn': 'turnover_rate'
}

target_columns = [
    'trade_date', 'stock_code', 'open', 'close', 'high', 'low',
    'volume', 'amount', 'pct_chg', 'turnover_rate'
]

failed_records = []

for code in tqdm(stock_code_lst, desc="Inserting daily market data"):
    try:
        _df = fetch_baostock_data(code, start_date, end_date)
        if _df is None or _df.empty:
            tqdm.write(f"No data for {code}, skip")
            continue

        _df = _df.reset_index()
        _df = _df.rename(columns=column_mapping)

        mapped_cols = list(column_mapping.values())
        present_cols = [col for col in mapped_cols if col in _df.columns]
        _df = _df[present_cols]

        _df['stock_code'] = code

        available_target = [col for col in target_columns if col in _df.columns]
        missing_target = set(target_columns) - set(available_target)
        if missing_target:
            tqdm.write(f"Warning: {code} missing columns {missing_target}, fill with NaN")
            for col in missing_target:
                _df[col] = None

        _df = _df[target_columns]

        _df['trade_date'] = pd.to_datetime(_df['trade_date']).dt.date
        _df['volume'] = _df['volume'].astype('Int64')

        try:
            client.insert_df('stock_daily_market', _df)
        except Exception as e:
            tqdm.write(f"Insert failed for {code}: {e}")
            failed_records.append((code, "insert", str(e)))

    except Exception as e:
        tqdm.write(f"Processing error for {code}: {e}")
        failed_records.append((code, "processing", str(e)))
        continue

if failed_records:
    print(f"\nTotal failures: {len(failed_records)} stocks")
    for code, stage, err in failed_records:
        print(f"  {code} ({stage}): {err}")
else:
    print("All stocks processed successfully.")

In [ ]:
%%sql
SELECT count(*) FROM quant.stock_daily_market